# Fine-tune Whisper for Sheng (Kenyan Swahili-English)**Runtime -> Change runtime type -> T4 GPU** before running.This notebook is a thin driver around the repo's scripts. The real logic lives in`src/whisper_sheng/` so it is testable and version-controlled, not trapped in cells.**Before you start**- Add `HF_TOKEN` in the Colab secrets panel (key icon, left sidebar).- Mount Drive so MLflow runs and checkpoints survive a disconnect.- **Read `progress.md` for the resume procedure.**

## 1. Check the GPUT4 is 15GB VRAM and has **no bf16 support**, which is why the config uses `fp16: true`.If you get an L4/A100 instead, enable bf16 with `--set train.fp16=false --set train.bf16=true`.

In [ ]:
!nvidia-smiimport torchprint("torch:", torch.__version__, "| cuda:", torch.version.cuda)print("gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")if torch.cuda.is_available():    props = torch.cuda.get_device_properties(0)    print(f"vram: {props.total_memory/1e9:.1f} GB | bf16 supported: {torch.cuda.is_bf16_supported()}")else:    raise SystemExit("No GPU. Runtime > Change runtime type > T4 GPU.")

## 2. Get the codeClone the repo. Edit `REPO_URL` if you forked it.

In [ ]:
import osREPO_URL = "https://github.com/lyle-mlengineer/audio-speech-ai.git"  # <-- edit if neededPROJECT = "audio-speech-ai"if not os.path.isdir(f"/content/{PROJECT}"):    !git clone {REPO_URL} /content/{PROJECT}%cd /content/{PROJECT}/finetune-whisper!ls

## 3. Credentials`HF_TOKEN` comes from Colab Secrets so it is never written into the notebook.

In [ ]:
from google.colab import userdataimport ostry:    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")    print("HF_TOKEN loaded from Colab secrets")except Exception as e:    print("No HF_TOKEN secret found:", e)from huggingface_hub import loginlogin(token=os.environ.get("HF_TOKEN"))

In [ ]:
# Colab already ships a matching torch; we only add our own deps on top.!pip install -q -r requirements.txtprint("deps installed")

## 4. Persist to Google DriveColab runtimes are wiped on disconnect. Mounting Drive keeps the MLflow databaseand checkpoints available when you reconnect.

In [ ]:
from google.colab import drivedrive.mount("/content/drive")DRIVE_DIR = "/content/drive/MyDrive/whisper-sheng"os.makedirs(DRIVE_DIR, exist_ok=True)print("Drive dir:", DRIVE_DIR)

## 5. Inspect the dataset42 rows, 1 speaker, all 30s clips. Seeing this matters: it sets honest expectationsfor what WER will mean here.

In [ ]:
from datasets import load_datasetds = load_dataset("lyle-mlengineer/kenyan-celebs", split="train")print(ds)print("columns:", ds.column_names)speakers = set(ds["speaker"])print("speakers:", speakers, "->", "SINGLE SPEAKER" if len(speakers) == 1 else "multi")row = ds[0]print()print("sample transcript:")print(row["text"][:300])print()print("audio array shape:", row["audio"]["array"].shape,      "| sr:", row["audio"]["sampling_rate"])

## 6. Smoke test (~2 min)Runs 4 steps on a handful of rows. Always do this before a real run: it catchesbroken deps, OOM at your batch size, and label/padding bugs cheaply.

In [ ]:
%cd /content/audio-speech-ai/finetune-whisper!python scripts/train.py --config configs/whisper_medium_t4.yaml --smoke-test --no-push --set train.output_dir={DRIVE_DIR}/smoke --set mlflow.tracking_uri=sqlite:///{DRIVE_DIR}/mlflow.db

## 7. Full fine-tuneOn a T4 expect roughly **1.5-2.5 hours** for whisper-medium at this effectivebatch size. `train.py` auto-resumes from the newest checkpoint in `output_dir`,so if Colab drops you mid-run, just re-run this same cell.

In [ ]:
%cd /content/audio-speech-ai/finetune-whisper!python scripts/train.py --config configs/whisper_medium_t4.yaml --run-name sheng-medium-t4 --set train.output_dir={DRIVE_DIR}/whisper-medium-sheng --set mlflow.tracking_uri=sqlite:///{DRIVE_DIR}/mlflow.db --set hub.repo_id=lyle-mlengineer/whisper-medium-sheng

## 8. Inspect results without the MLflow UIReading predictions directly is the fastest way to see whether Sheng slang isactually being learned, or whether the model just emits fluent Swahili.

In [ ]:
import jsonfrom pathlib import Pathreport_path = Path(f"{DRIVE_DIR}/whisper-medium-sheng/eval/eval_report.json")report = json.loads(report_path.read_text())print(f"WER={report['wer']:.4f}  CER={report['cer']:.4f}  "      f"mixedWER={report['mixed_wer']:.4f}  n={report['n_examples']}")print("=" * 70)for row in report["per_example"][:5]:    print(f"[WER {row['wer']:.3f}]")    print("REF:", row["reference"][:220])    print("HYP:", row["hypothesis"][:220])    print("-" * 70)

## 9. Confirm the Hub push

In [ ]:
from huggingface_hub import HfApiapi = HfApi()repo_id = "lyle-mlengineer/whisper-medium-sheng"print(f"{repo_id}:")for f in api.list_repo_files(repo_id):    print(" -", f)

## 10. Try the fine-tuned modelLanguage is pinned to `sw` because Whisper has no Sheng token. English insertionsstill come through, since only the decoder prompt is constrained.

In [ ]:
from transformers import pipelineasr = pipeline("automatic-speech-recognition",               model="lyle-mlengineer/whisper-medium-sheng",               device=0)row = ds[0]audio = {"array": row["audio"]["array"],         "sampling_rate": row["audio"]["sampling_rate"]}out = asr(audio, generate_kwargs={"language": "sw", "task": "transcribe"})print("PRED:", out["text"][:400])print()print("TRUE:", row["text"][:400])